## Init

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, lit
from pyspark.sql import functions as F

## Get timestamp of the last ingestion stored correctly

In [0]:
df = spark.sql("""
        SELECT 
            ROW_NUMBER() OVER (ORDER BY state_name) AS state_code,
            max(ingestion_timestamp) as last_successful_timestamp 
        FROM weather.bronze_data_raw
        GROUP BY state_name
    """)

In [0]:
df_timestamp = (
    df
    .withColumn(
        "last_successful_timestamp", 
        F.to_timestamp("last_successful_timestamp")
    )
)

In [0]:
from pyspark.sql.functions import from_json, col, explode, arrays_zip, to_timestamp, to_date, to_time, current_timestamp, max as spark_max, date_format
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, unix_timestamp
from pyspark.sql.functions import datediff, current_timestamp, col

df_elapsed_days = (
    df_timestamp
    .withColumn(
        "elapsed_days", 
        F.abs(datediff(current_timestamp(), col("last_successful_timestamp")))
    )
)

df_elapsed_hours = (
    df_elapsed_days
    .withColumn(
        "elapsed_hours", 
        ((unix_timestamp(current_timestamp()) - unix_timestamp(col("last_successful_timestamp"))) / 3600).cast("integer")
    )
)

## Create table with ingestion control by state/city

In [0]:
(
    df_elapsed_days.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("weather.ingestion_control")
)

## Sanity check

In [0]:
%sql
select * 
from weather.ingestion_control
limit 10